# Proyecto C-01_AGUIRRE: Repotenciación de la Planta Eólica Tejona
**Estudiante:** Emma Aguirre Torres  
**Ubicación del Proyecto:** Cordillera de Tilarán, Guanacaste, Costa Rica  
**Objetivo:** Desarrollar y evaluar el modelo técnico-económico para la modernización y repotenciación (*repowering*) de la Planta Eólica Tejona mediante análisis numérico, estadístico y financiero en Python.

---
### Estructura del Documento
1. **Sección 1:** Introducción y Contexto del Proyecto
2. **Sección 2:** Configuración del Entorno y Carga de Librerías
3. **Sección 3:** Ingestión y Validación de Datos (`datos/P09_C-01_AGUIRRE.xlsx`)
4. **Sección 4:** Limpieza y Saneamiento de Datos Meteorológicos
5. **Sección 5:** Termodinámica Atmosférica: Cálculo de Densidad del Aire
6. **Sección 6:** Métodos Numéricos Propios de Integración (Trapecio y Simpson)
7. **Sección 7:** Caracterización Estadística del Viento (Distribución de Weibull)
8. **Sección 8:** Curvas de Potencia de Aerogeneradores y Verificación del Límite de Betz
9. **Sección 9:** Proyección del Recurso en Altura y Escenarios de Repotenciación
10. **Sección 10:** Análisis Financiero y Valor Presente Neto (VPN)
11. **Sección 11:** Conclusiones Ejecutivas y Recomendaciones Técnicas

## 1. Introducción y Contexto del Proyecto

La **Planta Eólica Tejona**, ubicada en una de las regiones con mayor potencial eólico de Centroamérica (el paso de montaña entre el Volcán Arenal y la Cordillera de Guanacaste), ha operado exitosamente durante más de dos décadas. El régimen de vientos alisios acelerados por topografía compleja proporciona un recurso constante y de alta densidad energética.

El propósito de este estudio de repotenciación es evaluar el reemplazo de la flota existente de aerogeneradores por máquinas modernas de mayor altura de buje, mayor diámetro de rotor y electrónica de potencia avanzada. Este análisis integrará:
- Series temporales horarias de viento, presión y temperatura.
- Modelado de densidad y corrección según la norma internacional **IEC 61400-12-1**.
- Algoritmos numéricos propios para el cálculo de potencia media y energía neta anual.
- Evaluación de flujos de caja y rentabilidad de la inversión mediante **Valor Presente Neto (VPN)**.

## 2. Configuración del Entorno y Carga de Librerías

En esta sección se inicializan las bibliotecas científicas estándar (`numpy`, `pandas`, `matplotlib`, `plotly`, `scipy`) y se importan los módulos desarrollados en el paquete `src/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Importación de la arquitectura modular desarrollada en src/
from src.limpieza import cargar_datos, limpiar_viento, densidad_aire
from src.modelo import (
    TurbinaEolica,
    trapecio_propio,
    trapecio_acumulado,
    simpson_propio,
    potencia,
    energia,
    verificar_limite_betz,
    weibull_momentos,
    densidad_weibull,
    potencia_media,
)
from src.escenarios import escalar_viento_altura, energia_anual, vpn

%matplotlib inline
print("Entorno inicializado correctamente.")

## 3. Ingestión y Validación de Datos (`datos/P09_C-01_AGUIRRE.xlsx`)

Se carga el archivo Excel utilizando rutas relativas seguras mediante `pathlib.Path`. Se debe validar la existencia de las hojas mínimas necesarias y las columnas requeridas según el `diccionario_datos.md`.

In [ ]:
ruta_excel = Path("datos") / "P09_C-01_AGUIRRE.xlsx"

# Ejecutar una vez implementada la función en src/limpieza.py
# datos = cargar_datos(ruta_excel)
# print("Hojas cargadas:", list(datos.keys()))

## 4. Limpieza y Saneamiento de Datos Meteorológicos

Tratamiento sistemático de las series temporales de la estación anemométrica:
- Detección y filtrado de picos anómalos (*spikes*) de velocidad.
- Corrección de valores físicamente no válidos ($v < 0$ o $v > 35\text{ m/s}$).
- Imputación o interpolación controlada de valores faltantes (`NaN`).

In [ ]:
# df_viento_limpio = limpiar_viento(datos["Metereologia"])
# df_viento_limpio.describe()

## 5. Termodinámica Atmosférica: Cálculo de Densidad del Aire

Aplicación de la ecuación de estado de gas ideal para aire seco:
$$\rho = \frac{p_a}{R_{aire} \cdot T_K}$$

Donde $R_{aire} = 287.058\text{ J/(kg}\cdot\text{K)}$ y $T_K = T_C + 273.15$.

In [ ]:
# rho_serie = densidad_aire(
#     temperatura_c=df_viento_limpio["temperatura_c"],
#     presion_pa=df_viento_limpio["presion_pa"]
# )
# print(f"Densidad media estimada: {rho_serie.mean():.4f} kg/m^3")

## 6. Métodos Numéricos Propios de Integración (Trapecio y Simpson)

Verificación y prueba de las funciones numéricas desarrolladas sin librerías externas de cuadratura:
1. **Regla trapezoidal:** `trapecio_propio(y, x, dx)`
2. **Integral acumulativa:** `trapecio_acumulado(y, x, dx)`
3. **Regla de Simpson 1/3 compuesta:** `simpson_propio(y, x, dx)`

In [ ]:
# Verificación analítica rápida en celda
# x_test = np.linspace(0, np.pi, 101)
# y_test = np.sin(x_test)
# print("Integral de sin(x) en [0, pi] (Analítica = 2.0):")
# print("Trapecio:", trapecio_propio(y_test, x=x_test))
# print("Simpson:", simpson_propio(y_test, x=x_test))

## 7. Caracterización Estadística del Viento (Distribución de Weibull)

Ajuste de los parámetros de forma ($k$) y escala ($c$) mediante el método de los momentos:
$$\bar{v} = c \cdot \Gamma\left(1 + \frac{1}{k}\right)$$
$$\sigma_v^2 = c^2 \left[ \Gamma\left(1 + \frac{2}{k}\right) - \Gamma^2\left(1 + \frac{1}{k}\right) \right]$$

In [ ]:
# k_est, c_est = weibull_momentos(df_viento_limpio["velocidad_viento"].values)
# print(f"Parámetros de Weibull estimados: k = {k_est:.3f}, c = {c_est:.3f} m/s")

## 8. Curvas de Potencia de Aerogeneradores y Verificación del Límite de Betz

Evaluación de la aerodinámica del rotor:
$$C_p(v) = \frac{P_{electrica}(v)}{\frac{1}{2} \rho_{ref} A v^3} \le \frac{16}{27} \approx 0.5926$$

Corrección de potencia por densidad del aire según IEC 61400-12-1.

In [ ]:
# cp_existente = verificar_limite_betz(curva_existente, area_existente)
# cp_nueva = verificar_limite_betz(curva_nueva, area_nueva)
# print("¿Ambas curvas respetan el límite de Betz?:", (cp_existente.max() <= 16/27) and (cp_nueva.max() <= 16/27))

## 9. Proyección del Recurso en Altura y Escenarios de Repotenciación

Extrapolación del perfil de velocidad vertical con la ley de Hellmann:
$$v(h_{nueva}) = v_{ref} \cdot \left(\frac{h_{nueva}}{h_{ref}}\right)^\alpha$$

Cálculo de la producción neta anual de energía (AEP) aplicando factores de pérdidas globales de parque.

In [ ]:
# v_80m = escalar_viento_altura(df_viento_limpio["velocidad_viento"], h_ref=40.0, h_nueva=80.0, alpha=0.14)
# aep_existente = energia_anual(potencia_media_existente, n_turbinas=12, perdidas_l=0.15)
# aep_repotenciada = energia_anual(potencia_media_nueva, n_turbinas=6, perdidas_l=0.12)

## 10. Análisis Financiero y Valor Presente Neto (VPN)

Evaluación de la viabilidad económica incremental de la repotenciación:
$$\text{VPN} = -\text{CAPEX} + \sum_{t=1}^{N} \frac{(\Delta E_{anual} \cdot \text{Precio}_{kWh}) - \Delta\text{OPEX}}{(1 + r)^t}$$

In [ ]:
# vpn_proyecto = vpn(
#     inversion=capex_usd,
#     delta_energia_kwh=(aep_repotenciada - aep_existente),
#     precio_kwh=tarifa_usd_kwh,
#     delta_mantenimiento=delta_opex_usd,
#     tasa_descuento=0.08,
#     anios=20
# )
# print(f"Valor Presente Neto (VPN): ${vpn_proyecto:,.2f} USD")

## 11. Conclusiones Ejecutivas y Recomendaciones Técnicas

En esta sección final la estudiante sintetizará los hallazgos del estudio:
1. Incremento porcentual en la captura energética neta (MWh/año).
2. Comparación del factor de planta (CF) antes y después de la repotenciación.
3. Rentabilidad económica según el criterio del VPN y sensibilidad a la tarifa energética.
4. Recomendación técnica fundada sobre la conveniencia de la modernización de la Planta Eólica Tejona.